# Drone Detection & Tracking — one YOLO11n on **eight** datasets, on Kaggle T4×2

This notebook answers the question the single-dataset notebooks cannot: **is one model
trained on everything better than a model trained on the best single dataset?**

It acquires all eight sources, harmonises them into one two-class label space, builds a
single combined dataset, trains **one** YOLO11n on it, and evaluates it against every
source separately.

### Why one model on many datasets is the right shape for this system

The deployed system watches **100 cameras** and has to generalise to sites, sensors and
weather it has never seen. A model trained on one dataset is a model that has seen one
distribution. The argument for combining is not aesthetic:

| | single dataset | all eight |
| --- | --- | --- |
| drone appearance | one site's lighting, one drone type | eight sources' worth of shapes and scales |
| target scale | one median (28 px, or 12 px, or 171 px) | 12 px to 171 px in one model |
| bird rejection | only if *that* dataset has birds | three bird-negative sources contribute |
| modality | one (RGB or IR) | RGB **and** thermal IR |

### The honest caveat, stated up front

A combined mAP is **not** the number to report. The eight sources disagree about what a
hard sample is: MM-UAV's 12 px targets dominate the loss while `dvb`'s birds dominate the
precision. One aggregate number over all of them is close to meaningless.

**The per-dataset cross-evaluation table is the result.** `evaluate --cross` trains once and
scores each source separately, with a falsifiability note on every row, and that table is
what you compare against the single-dataset runs.

### What you should see

| Stage | Command | Approx. cost |
| --- | --- | --- |
| Verify | `anti-uav verify-env` | seconds |
| Weights | `anti-uav fetch-weights` | ~6 MB |
| Data | `anti-uav download --dataset all` | ~20 GB (the light sources) |
| Preprocess | `convert` → `splits` → `build` → `sanity` | ~45–90 min |
| Train | `anti-uav train --model yolo11n --combo all8` | **~3–6 h** (120 epochs) |
| Evaluate | `anti-uav evaluate --cross` | ~10 min |

**`antiuav` and `mmuav` are skipped by default here.** `antiuav` is a 60 GB Google Drive
transfer and `mmuav` is Baidu-Pan-only behind an interactive captcha that cannot be solved
from a notebook. The combo below uses the six sources a Kaggle session can actually fetch.
Uncomment the two lines to include them if you have the credentials and a captcha solver.

---

## Before you run

**Settings → Accelerator → GPU T4 x2.** Kaggle restarts the session when you change this,
so do it before running anything.

**Settings → Internet → On.** Required three times over: the `git clone`, the `uv` install,
and the dataset downloads.

**Add a Kaggle API token as a notebook secret** (Settings → Secrets, or Add-ons →
Secrets). One of these works:

| Secret name | Value |
| --- | --- |
| `KAGGLE_API_TOKEN` | the token from Account → Settings → API — **preferred** |
| `KAGGLE_USERNAME` + `KAGGLE_KEY` | account name and legacy API key, both required |

`KAGGLE_API_TOKEN` is the modern path and the one Kaggle's UI hands you by default. The
username/key pair is the older shape and still authenticates, but both halves must be set —
one alone is not a credential, and the download will refuse with a message naming what is
missing rather than handing you an opaque 401 from the CLI.

Two of the eight sources are Kaggle-hosted and need this token. The other four
(`github_maciullo`, `hf_seraphim`, and the two Kaggle ones) need nothing at all — so if the
token is missing, the run still works, just with fewer sources in it.

One Kaggle quirk worth knowing in advance: you may also need to **accept the rules** for
`romsham/dronevsbird-foryolo`, `cybersimar08/drone-detection` and
`banderastepan/drone-detection` in a browser tab. Kaggle gates some datasets behind a rules
acceptance that the API cannot satisfy for you, and the error it returns is a 403 with no
explanation.

---

In [ ]:
# --- 1. Clone -------------------------------------------------------------

REPO_URL = "https://github.com/uzairlol/drone-detection-tracking.git"
REPO_DIR = "/kaggle/working/anti-uav"

!git clone --depth 1 {REPO_URL} {REPO_DIR}

In [ ]:
# --- 2. Install -----------------------------------------------------------

# `uv` rather than `pip`: same resolution, a fraction of the wall clock, and it leaves an
# already-satisfied requirement alone. That last part is the point - Kaggle ships a
# CUDA-matched torch/numpy pair, and replacing it with a different build is the classic way
# to break `import torch` on a GPU that works fine.
#
# `--system` installs into the image's interpreter. A venv would need its bin/ prepended to
# PATH in every subsequent cell, which is one forgotten cell away from a silent ImportError.

!pip install --quiet uv
!uv pip install --system -e "{REPO_DIR}[train,download]"

### Reading the slow cells

Every command after this one runs through a small `run()` helper instead of a bare `!` shell
escape. It does three things worth having:

- **Echoes the command first**, so the cell you are looking at and the command that actually
  ran cannot be different.
- **Streams output line by line.** A 3-hour training run is then visible progress rather than
  a spinning indicator, and ultralytics' per-epoch mAP is readable as it climbs.
- **Reports wall clock, including on failure** — `[ok] 3h 12m` or `[EXIT 1] 6m 03s`. Knowing
  how far it got before dying is usually the most useful thing in the output.

`build` and `sanity` are also *gated* now: the cell raises and stops rather than carrying on
into a training run with a leaked split.

### Scratch

Everything writable goes to `/kaggle/tmp`, never `/kaggle/working`.

This is a hard limit, not a preference: only **20 GB** of `/kaggle/working` is autosaved per
notebook version, while the raw datasets plus the interim and processed trees come to well
over that. `/kaggle/tmp` is scratch — far larger, and discarded with the session.

`ANTI_UAV_DATA_DIR` moves the `data/` and `artifacts/` trees while `configs/` is still read
from the clone, so not one pipeline behaviour changes. It must be set **before** the first
`anti-uav` call, because path resolution is cached on first use.

Then read `verify-env` rather than trusting the exit code alone. Three lines carry the
meaning:

| Line | Must say | If it does not |
| --- | --- | --- |
| `profile` | `turing` | `pascal` means auto-detection did not run and **AMP is off** |
| `python` | `[ok]` | Kaggle ships 3.13, which is supported. 3.14+ is refused as untested |
| `arch-kernels` | `[ok] … sm_75 present` | Kaggle's torch lacks Turing kernels; nothing else will help |
| `base-weights` | warn, not fail | expected — `*.pt` is gitignored; cell 5 fetches it |

Two GPUs is also worth confirming here: you should see two `device` / `arch-kernels` pairs,
one per T4. If only one appears you have a single-GPU session, and `--device 0,1` will fail.

This step exists because the failure it catches is silent. A torch wheel built without your
card's architecture imports cleanly, reports `torch.cuda.is_available() == True`, and then
dies hours in with `no kernel image is available for execution on the device`. Checking
availability is not enough; the arch list has to be checked too.

In [ ]:
import os

os.environ["ANTI_UAV_DATA_DIR"] = "/kaggle/tmp/anti-uav"

# Auto-detect is correct on a T4 now (sm_75 -> turing). Pinned anyway, so the notebook cannot
# silently pick a different profile if the accelerator is switched to a P100 (sm_60).
os.environ["ANTI_UAV_PROFILE"] = "turing"

# A small runner for the slow cells. It prints the command first, streams its output so a
# long run is not a silent spinning dot, and reports wall clock either way - including when
# something fails, which is when you most want to know how far it got before dying.
import shutil
import subprocess
import time

def run(*argv, note=""):
    """Run a CLI command, streaming output, with a wall-clock report."""
    import shutil as _shutil

    if _shutil.which(argv[0]) is None:
        raise SystemExit(
            f"{argv[0]!r} is not on PATH. If this is the anti-uav command, the\n"
            f"editable install did not land - re-run the install cell."
        )
    print(f"\n\033[1m$ {' '.join(argv)}\033[0m")
    if note:
        print(f"  {note}")
    started = time.monotonic()
    proc = subprocess.Popen(argv, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=os.environ.copy())
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    elapsed = time.monotonic() - started
    mins, secs = divmod(elapsed, 60)
    verdict = "ok" if proc.returncode == 0 else f"EXIT {proc.returncode}"
    print(f"\033[1m  [{verdict}] {mins:.0f}m {secs:04.1f}s\033[0m")
    return proc.returncode

def anti_uav(*args, note=""):
    return run("anti-uav", *args, note=note)

print("ANTI_UAV_DATA_DIR ->", os.environ["ANTI_UAV_DATA_DIR"])
print("anti-uav on PATH ->", shutil.which("anti-uav") or "NOT FOUND")

anti_uav("verify-env", note="expect: profile turing, 2x sm_75, and no FAIL")

## 4. Acquire — all the sources this session can reach

**Weights first, on purpose.** ultralytics downloads a missing base checkpoint from *inside*
the trainer, so on a slow or blocked network the run dies *after* the dataset is built and the
run directory created. Fetching here moves that failure to a five-second one.

**The list below is the whole point of this notebook.** Six sources, fetched one at a time so
a single failed transfer does not take the run down with it.

| source | transport | auth | ~size |
| --- | --- | --- | --- |
| `dvb` | Kaggle | token | 14 GB |
| `mavvid` | Bitbucket | none | 6 GB |
| `github_maciullo` | git clone | none | 3 GB |
| `hf_seraphim` | HuggingFace | none | 4 GB |
| `kaggle_cybersimar` | Kaggle | token | 2.5 GB |
| `kaggle_banderastepan` | Kaggle | token | 5 GB |
| `antiuav` | Google Drive | none | 60 GB |
| `mmuav` | Baidu Pan | captcha | 380 GB |

`antiuav` and `mmuav` are commented out in the next cell — 440 GB and a captcha are not a
Kaggle-notebook proposition. Uncomment them if you have the bandwidth and a solver.

In [ ]:
# Prefer KAGGLE_API_TOKEN; fall back to the username/key pair. Whichever is
# present, `download` names the shape it found rather than just saying yes/no.
from kaggle_secrets import UserSecretsClient

_secrets = UserSecretsClient()
_loaded = []

for _name in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
    try:
        os.environ[_name] = _secrets.get_secret(_name)
        _loaded.append(_name)
    except Exception:
        pass

print(f"credentials loaded: {', '.join(_loaded) or 'NONE'}")

anti_uav("fetch-weights", "--models", "yolo11n", note="~6 MB")

In [ ]:
# The six sources a Kaggle session can actually reach. One at a time on purpose: a
# single failed transfer should not take the whole run down with it.
#
# `dvb` is 14 GB of Kaggle mirror and the most valuable source in the set (it is one of
# only three with bird negatives, so it is what makes the precision number mean something).
SOURCES = [
    "dvb",
    "mavvid",
    "github_maciullo",
    "hf_seraphim",
    "kaggle_cybersimar",
    "kaggle_banderastepan",
]

# Uncomment if you have the bandwidth and (for mmuav) a captcha solver. 440 GB total.
# SOURCES += ["antiuav"]
# SOURCES += ["mmuav"]

for _ds in SOURCES:
    anti_uav("download", "--dataset", _ds, "--dry-run", note="free - what would be fetched")
    print()


In [ ]:
# ~35 GB total across the six sources. Streams the CLI's progress as it goes.
#
# A failure here is NOT fatal to the run: `build` skips a source that is missing and
# records it as `skipped` in build_report.json. The combo still trains, on fewer datasets.
# But check that report before believing a combo name is evidence of dataset membership -
# `build` will happily name the combo after a source that never arrived.
for _ds in SOURCES:
    anti_uav("download", "--dataset", _ds, note="several minutes each")
    print()


## 5. Preprocess — and why the order is not negotiable

```
convert  →  splits  →  build  →  sanity
```

`build` reads each frame's split field off the interim index. Build before splitting and you
materialise an **empty train set**, then `sanity` fails — correctly, but after wasting the
copy.

**`splits` assigns whole sequences, never individual frames.** Every source here is
video-derived, so adjacent frames are near-identical. Split at the frame level and the same
target's near-duplicates land on both sides of the boundary: mAP inflates by double digits and
the number is meaningless. `sanity` fails the build if any sequence appears in two splits, so
this is enforced rather than merely documented.

**Tiling is decided per source, not per combo.** `dvb`'s 28 px median target is below the
40 px threshold, so it is tiled at 640 px. `mavvid`'s 171 px targets and the three new
sources' 56–80 px targets are not. Tiling the whole combo would penalise the easy sources
for no reason, and cost is paid only where recall lives.

### What `stats` will tell you

The class-coverage table is the cell to read carefully. It is the one that says which sources
carry bird annotations and which do not — and therefore which precision numbers in the final
table are evidence and which are decoration.

In [ ]:
for _ds in SOURCES:
    anti_uav("convert", "--dataset", _ds, note="~40k frames each, several minutes")
    anti_uav("stats", "--dataset", _ds, note="~1 min")
    print()


In [ ]:
# convert -> splits -> build -> sanity. The order is load-bearing: `build` reads each
# frame's split field off the interim index, so building first gives an empty train set.
# `sanity` exits non-zero on failure - read its output, do not just scroll past it.

# The combo name is the matrix slug. `all8` is the full eight-source combo; for the six
# sources this notebook can actually fetch, use the six-source slug below so the build
# report matches what is on disk.
COMBO = "dvb+mavvid+github_maciullo+kaggle_cybersimar+hf_seraphim+kaggle_banderastepan"

if anti_uav("splits", "--combo", COMBO, note="seconds") != 0:
    raise SystemExit("splits failed - do not continue, build would produce an empty train set")

anti_uav("build", "--combo", COMBO, note="~45-90 min, writes the whole combined set")

# Last gate. Non-zero here means a leaked split or a failed invariant - stop.
if anti_uav("sanity", "--combo", COMBO, note="hard gate") != 0:
    raise SystemExit("sanity FAILED - do not train on this build")

## 6. Train — one model, all datasets

Dry-run first, every time. It prints the resolved hyperparameters, the output path and the
dataset summary without allocating a GPU — the cheapest possible way to catch a wrong combo,
and the only place the resolved batch is visible before it either works or OOMs.

```
batch  : 32        ← a concrete integer; a fraction here aborts under DDP
amp    : True      ← if False, the turing profile did not load
```

### Reading the batch under DDP

The printed batch is the **total across both GPUs**, so 32 means 16 per rank. It is a fixed
integer rather than AutoBatch for two reasons: ultralytics resolves `batch: -1` to a
*fraction* of free VRAM and then raises outright when `world_size > 1`, and DDP splits the
total evenly, so an odd number would leave a rank idle.

### Performance expectation

**~3–6 h for 120 epochs on T4×2.** Expect a **sub-2× speedup from two GPUs**, not 2×: at
this dataset size the bottleneck is four CPU cores decoding 640 px JPEG, not the cards.

**If you OOM, halve the batch before changing anything else.** A combined dataset is bigger
than any single one, so OOM is more likely here than in the single-dataset notebooks.

**If you are short on time**, `--epochs 30` gives a usable model: mAP is meaningful well
before 120 epochs, and you get the whole pipeline on the record for a quarter of the GPU
time.

In [ ]:
anti_uav("train", "--model", "yolo11n", "--combo", COMBO,
         "--profile", "turing", "--device", "0,1", "--workers", "2",
         "--dry-run",
         note="dry run: confirms batch 32 / amp True before committing GPU hours")

### YOLO11n — the edge candidate

2.6 M parameters, which is what makes it the deployable option: small enough to sit behind
`nvinfer` on a Jetson against a ~3 ms budget. The matrix's other family, RT-DETR-x2, is the
accuracy reference and is not run here.

### The augmentation question this run answers

Combining eight sources means eight different cameras, lighting conditions and drone
types in one training set. The trainer's augmentation is what stops the model from
memorising "this particular sky, this particular resolution" instead of learning "drone".
That is the mechanism by which this run should beat the single-dataset ones — not by having
more data, but by having *diverse* data that defeats a shortcut.

In [ ]:
# ~3-6 h for 120 epochs. Add --epochs 30 for a fast pass (useful, and 4x cheaper).
#
# FALLBACK: if DDP cannot spawn inside the notebook kernel, re-run with --device 0
# (single GPU, same batch). Identical results, similar wall clock, no subprocess to
# go wrong. The preflight warns before training if you ask for more GPUs than are present.
#
# This is the long one. `run()` streams ultralytics' per-epoch lines, so you can watch
# it progress and see mAP climb rather than staring at a spinning indicator for hours.
anti_uav(
    "train", "--model", "yolo11n", "--combo", COMBO,
    "--profile", "turing", "--device", "0,1", "--workers", "2",
    note="3-6 h for 120 epochs - this is the one to leave running",
)

### Reading the run

`list-runs` reads the run directory off disk and prints run, model, epochs, mAP50,
mAP50-95, precision, recall and wall clock.

**mAP50-95 is the headline, not mAP50.** At a 28 px median target mAP50 is generous — a
detector that puts a loose box on the right patch scores well. The stricter metric is the
one that discriminates between a detector that found the bird and one that found a patch of
sky.

## 7. Evaluate — the per-dataset table is the result

`--cross` is the number worth showing, and it is the **reason this notebook exists**.

A single combined mAP across heterogeneous datasets is close to meaningless, because the
sources disagree about what a hard sample is — `dvb`'s 28 px targets dominate the loss
while `dvb`'s birds dominate the precision, and the three drone-only sources contribute
neither difficulty nor evidence. The per-dataset table is the honest one.

### Read the falsifiability note, not just the number

The evaluator prints a caveat next to any precision figure whose dataset cannot support the
claim:

> for `antiuav` and `mmuav`, precision cannot falsify a false-positive claim — neither
> contains bird annotations, so a detector that called every bird a drone would score
> perfectly.

Three of the eight rows say **yes**:

| source | birds? | precision falsifiable? | what its row tells you |
| --- | --- | --- | --- |
| `dvb` | yes | **yes** | bird rejection on the hardest source |
| `mavvid` | yes | **yes** | bird rejection on large, easy targets |
| `github_maciullo` | yes | **yes** | bird rejection on a third distribution |
| `kaggle_cybersimar` | no | no | drone recall on one distribution |
| `hf_seraphim` | no | no | drone recall on another |
| `kaggle_banderastepan` | no | no | drone recall on a third |
| `antiuav` | no | no | drone recall, IR modality (not in this run) |
| `mmuav` | no | no | drone recall, 12 px targets (not in this run) |

**The comparison to make** is this table against the single-dataset runs' tables. The
question is not "is the combined mAP higher" — it is "is the combined model at least as
good on the *bird-negative* sources, and better on the sources it has never seen?".

That second half is the whole hypothesis. A model trained on `dvb` alone will score badly
on `github_maciullo`, and a model trained on all of them should not.

In [ ]:
anti_uav("evaluate", "--run", f"artifacts/runs/yolo11n/yolo11n__{COMBO}", "--cross",
         note="~10 min - this is the result")

## 8. Why `track-eval` refuses most of these datasets

Run it and it declines. **That is correct behaviour, and worth demonstrating rather than
working around** — it is the tooling refusing to report a number it cannot stand behind.

| Dataset | Bird negatives | MOT ground truth | Multi-object identity | Verdict |
| --- | --- | --- | --- | --- |
| `dvb` | **yes** | no | no | detection metrics only |
| `mavvid` | **yes** | no | no | detection metrics only |
| `github_maciullo` | **yes** | no | no | detection metrics only |
| `antiuav` | no | yes | **no — single target** | MOTA/HOTA only; IDF1 and IDSW are vacuous |
| `mmuav` | no | yes | **yes** | everything, including IDF1 |
| `kaggle_cybersimar` | no | no | no | detection metrics only |
| `hf_seraphim` | no | no | no | detection metrics only |
| `kaggle_banderastepan` | no | no | no | detection metrics only |

The subtle row is `antiuav`. It *has* track ids, so a naive `has_track_ids` check passes —
but every box is id 1, so every identity trivially matches every other. IDF1 would report a
perfect score for a tracker doing nothing. That is why `capabilities.py` derives an explicit
contract per source rather than trusting a single boolean, and why `track-eval` **warns** on
single-target ground truth instead of refusing it.

Only `mmuav` makes IDF1 meaningful, and it is Baidu-Pan-only with an interactive captcha —
not something to attempt live. The trackers themselves run fine on every source; there is
simply nothing to score them against.

In [ ]:
anti_uav("track-eval", "--run", f"artifacts/runs/yolo11n/yolo11n__{COMBO}", "--dataset", "dvb",
         note="declines by design: dvb has no track ids")

## 9. Inference and the rule layer

**This project's own trackers, not ultralytics'.** Version 8.4 ships no `track` task —
`persist` and `tracker` are not in the predict config, and `model.predict(..., persist=True)`
raises `SyntaxError`. Owning SORT / ByteTrack / BoT-SORT buys something more valuable: the
live path and the `track-eval` harness share **one** association implementation, so a number
from the harness describes what the pipeline actually does.

### Why inference is permissive and alerting is strict

Two different stages, two different jobs:

| | threshold | why |
| --- | --- | --- |
| detector | `conf_threshold: 0.25` | maximise **recall** — the tracker needs the low-confidence frames to carry motion evidence through a dip |
| rules | `confidence.initiate: 0.60` | maximise **precision** — buy it *after* tracking, using persistence, kinematics and cross-camera agreement |

Raise the detector threshold to 0.60 and the tracker starves. `predict` therefore prints the
per-detection table rather than a count, because 40 detections at conf 0.25 and 3 at 0.8 are
completely different situations.

### The five rule gates

`configs/rules/drone_rules.yaml`, in order: **confidence** (with hysteresis), **persistence**,
**kinematics**, **spatial**, **cross-camera**. Every threshold is tunable without a retrain.

Two of them return `skipped` rather than `passing` until the site is calibrated —
`spatial.horizon_y` and `ground_plane_z_m` are null, and they are per-camera properties.
That distinction is the point of the layer: **a missing input must never read as a satisfied
rule.** Only `FAIL` blocks an alert; an uncalibrated site runs three of five gates and says
plainly which two it skipped and what to configure.

Note `cross_camera.min_cameras: 2`. One view cannot distinguish a drone from a bird with
enough confidence to page anyone — so a track with no global identity is a **failure**, not a
skip. This is why cross-camera fusion is load-bearing for correctness rather than a nice
extra.

In [ ]:
# Grab a real frame from the interim index so the cell is self-contained. Swap --source for
# a video clip to show tracker behaviour over time - that is the more convincing demo, since
# a single frame cannot show a track being held through a dip or broken by an occlusion.
import glob

frames = sorted(
    glob.glob(f"{os.environ['ANTI_UAV_DATA_DIR']}/data/interim/*/*/frames/*/rgb/*.jpg")
)
print(f"{len(frames):,} interim frames available across all sources")

anti_uav("predict", "--run", f"artifacts/runs/yolo11n/yolo11n__{COMBO}",
         "--source", frames[len(frames) // 2], "--tracker", "botsort",
         "--save", "/kaggle/tmp/preds", note="seconds")

In [ ]:
anti_uav("rules", "show")
anti_uav("rules", "validate", note="cross-rule tensions; advisory")

## 10. Keep the evidence

`artifacts/` lives under `/kaggle/tmp` and is **not** autosaved — the weights would vanish with
the session. Copy the record out to `/kaggle/working`, which is. That single step is the
difference between a demo and a result you can point at afterwards.

A run directory is the record: `best.pt`, `results.csv`, `args.yaml` and a metadata file.
**Copy it, do not regenerate it.** `data/processed/<combo>/build_report.json` is worth
keeping too — it is what proves which sources were actually present, and `build` will happily
skip a missing one and still name the combo after it. A combo name is not evidence of dataset
membership.

In [ ]:
import shutil
from pathlib import Path

SRC = Path(os.environ["ANTI_UAV_DATA_DIR"])
DST = Path("/kaggle/working/artifacts")

for _rel in ("runs", "exports", "tracks"):
    _src = SRC / "artifacts" / _rel
    if _src.exists():
        shutil.copytree(_src, DST / _rel, dirs_exist_ok=True)
        print(f"saved artifacts/{_rel}")

_report = SRC / "data" / "processed" / COMBO / "build_report.json"
if _report.is_file():
    (DST / "combined_build_report.json").write_text(_report.read_text())
    print("saved combined_build_report.json")
else:
    print("no build_report.json - CHECK WHICH SOURCES ACTUALLY LANDED")

In [ ]:
anti_uav("list-runs")

## What this notebook deliberately does not do

**`antiuav` and `mmuav`.** 60 GB and 380 GB, the latter behind an interactive Baidu captcha.
Neither is a Kaggle-notebook proposition. The code path for them is identical to the other
six; only the acquisition is the problem.

**`anti-uav serve`** — the operator console and JSON API. Kaggle tears the session down when
the last cell finishes and does not forward a listening port, so a long-lived server has
nowhere to live. Run it locally against the copied `artifacts/`.

**TensorRT export** — the engine is normally built on the target device, not on a training
box, so `deploy render` *warns* rather than fails when the engine is missing. ONNX export
works if you want it.

**The other 21 runs** — the matrix is 2 models × 11 combos; this notebook does 1 of the 22.
T4×2 proves the pipeline; it does not sweep it.

**Tracking metrics** — see section 8. Only `mmuav` makes them meaningful.

---

## What one run does and does not establish

Eight public datasets, one architecture, one combined combo, one site model. This run
answers **"does one model trained on everything generalise better than a model trained on
the best single dataset, and does it still reject birds on the sources that can test that"**.

It does not answer "how well does this detect an unseen drone type at 40 m against a
cluttered sky". Eight datasets is a design choice made under a time budget, not a claim of
coverage. **Treat the ranking as a ranking and the absolute numbers as specific to these
eight sources.**

The strongest claim here is a negative one, which is the useful kind: whether the detector
fires on birds. Everything finer needs the full matrix on real hardware.